# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the `mlcroissant` library, accessing structured biomedical data through its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Title: {metadata.name}\n\nDescription: {metadata.description}")
print(f"Croissant Schema version: {metadata.conformsTo}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.
This section will enumerate all record sets and their fields, referencing by `@id` as required.

In [ ]:
# Access top-level record sets, fields, and columns by `@id`
records_metadata = dataset.metadata.recordSet

print(f"Found {len(records_metadata)} record sets.\n")
for rs in records_metadata:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"Name: {rs.get('name', '(no name)')}")
    print(f"Description: {rs.get('description', '(no description)')}")
    fields = rs.get('field', [])
    # Ensure fields is always a list for uniform iteration
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        print(f"    - @id: {f['@id']} ({f.get('name', f.get('@id'))})")
    print('-' * 50)


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references are made **explicitly using `@id` fields**.

***Note:*** The actual `@id`s for record sets and fields depend on the dataset metadata. We'll use these dynamically, following the FAIR² Croissant schema.

In [ ]:
# Prepare a list of all record set `@id`s
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    # Load all records for this set (iterator of dicts)
    records_iter = dataset.records(record_set=record_set_id)
    # Expand to list, store in DataFrame
    records = list(records_iter)
    if not records:
        print(f"  No records found for {record_set_id}\n")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}\n")

# Display available DataFrames
print(f"Available record set IDs: {list(dataframes.keys())}")
# Pick the first record set as example
example_record_set_id = next(iter(dataframes))
print(f"Example record set: {example_record_set_id}")
print("Columns:", dataframes[example_record_set_id].columns.tolist())
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process a numeric field from the example record set, applying filtering and normalization, referencing fields by their `@id`.

In [ ]:
# Select an example numeric field by `@id` from the chosen record set
df = dataframes[example_record_set_id]

# Try to automatically pick a numeric column for demo
numeric_candidates = df.select_dtypes(include=['float64','int64']).columns
if len(numeric_candidates) == 0:
    # If none, do a best effort to find something with 'count', 'num', 'abundance', or 'MW' in the name
    for col in df.columns:
        if any(w in col.lower() for w in ['count', 'num', 'abundance', 'mw', 'mass', 'value', 'coverage']):
            numeric_candidates = [col]
            break

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Selected numeric field for EDA: {numeric_field_id}")
    # Try conversion to float if needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
else:
    raise ValueError("No numeric field found in example record set - please adjust field selection.")

threshold = df[numeric_field_id].median() # Use median as threshold for demonstration
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}\n")

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a group field if possible
group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < 10]
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"Grouping by field: {group_field_id}\n")
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    )
    print(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Let's visualize the filtered and normalized data for the selected numeric field.

For demonstration, we'll plot a histogram of the normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=20)
plt.title(f"Distribution of normalized values for {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Count")
plt.grid(True)
plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a Croissant-structured biomedical dataset and explore its metadata
- Reference all entities by their schema `@id` as best practice
- Load records from a chosen record set and perform basic EDA
- Filter and normalize a numeric field
- Visualize the data distribution

This workflow can be extended for deeper bioinformatics or machine learning analysis using the FAIR² dataset.